In [1]:
import pandas as pd
import joblib

# Load the saved tuned model
best_lr_model = joblib.load('final_model.pkl')

# Load test data fresh
X_test = pd.read_csv('../Data/X_test.csv')
y_test = pd.read_csv('../Data/y_test.csv')
y_test = y_test['payment_recovered']  # flatten to Series, same fix as before

print(type(best_lr_model))
print(X_test.shape, y_test.shape)

<class 'sklearn.linear_model._logistic.LogisticRegression'>
(1409, 17) (1409,)


In [2]:
y_proba_tuned = best_lr_model.predict_proba(X_test)[:, 1]
print(y_proba_tuned[:5])

[0.49849078 0.66565963 0.12750029 0.61779278 0.12289599]


In [3]:
def recommend_action(probability, failure_reason, customer_segment, retry_count):
    
    if failure_reason == 'card_expired':
        action = 'switch_payment_method'
        explanation = f"Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead."
    
    elif failure_reason == 'insufficient_funds' and probability >= 0.4:
        action = 'delayed_retry'
        explanation = f"Failure reason is insufficient_funds with a reasonable recovery probability ({probability:.2f}). Waiting allows time for the customer's balance to potentially refill before retrying."
    
    elif probability >= 0.7:
        action = 'immediate_retry'
        explanation = f"High recovery probability ({probability:.2f}) with no time-dependent failure reason. Retrying immediately is likely to succeed."
    
    elif probability < 0.4 and customer_segment == 'high_value_repeat':
        action = 'send_incentive'
        explanation = f"Low recovery probability ({probability:.2f}), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment."
    
    else:
        action = 'delayed_retry'
        explanation = f"No strong signal for immediate action (probability={probability:.2f}). Defaulting to a low-cost delayed retry rather than an expensive incentive."
    
    return action, explanation

In [4]:
print(recommend_action(0.9, 'card_expired', 'occasional', 1))
print(recommend_action(0.8, 'insufficient_funds', 'occasional', 0))
print(recommend_action(0.85, 'network_error', 'new_customer', 0))
print(recommend_action(0.25, 'bank_decline', 'high_value_repeat', 1))
print(recommend_action(0.2, 'bank_decline', 'new_customer', 2))

('switch_payment_method', 'Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead.')
('delayed_retry', "Failure reason is insufficient_funds with a reasonable recovery probability (0.80). Waiting allows time for the customer's balance to potentially refill before retrying.")
('immediate_retry', 'High recovery probability (0.85) with no time-dependent failure reason. Retrying immediately is likely to succeed.')
('send_incentive', 'Low recovery probability (0.25), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment.')
('delayed_retry', 'No strong signal for immediate action (probability=0.20). Defaulting to a low-cost delayed retry rather than an expensive incentive.')


In [5]:
segment_cols = ['customer_segment_high_value_repeat', 
                'customer_segment_new_customer', 
                'customer_segment_occasional']

X_test['customer_segment_decoded'] = X_test[segment_cols].idxmax(axis=1)
X_test['customer_segment_decoded'] = X_test['customer_segment_decoded'].str.replace('customer_segment_', '')

print(X_test['customer_segment_decoded'].value_counts())

customer_segment_decoded
high_value_repeat    754
occasional           414
new_customer         241
Name: count, dtype: int64


In [6]:
failure_cols = ['failure_reason_bank_decline', 'failure_reason_card_expired', 
                'failure_reason_insufficient_funds', 'failure_reason_network_error']

X_test['failure_reason_decoded'] = X_test[failure_cols].idxmax(axis=1)
X_test['failure_reason_decoded'] = X_test['failure_reason_decoded'].str.replace('failure_reason_', '')

print(X_test['failure_reason_decoded'].value_counts())

failure_reason_decoded
insufficient_funds    630
bank_decline          281
card_expired          278
network_error         220
Name: count, dtype: int64


In [7]:
X_test['probability'] = y_proba_tuned

In [8]:
print('probability' in X_test.columns)
print(X_test.shape)

True
(1409, 20)


In [9]:
def map_action_to_channel(action):
    channel_map = {
        'immediate_retry': 'payment_gateway_auto_retry',
        'delayed_retry': 'payment_gateway_scheduled_retry',
        'send_incentive': 'sms_and_email_offer',
        'switch_payment_method': 'sms_prompt_update_method',
        'escalate_to_manual_review': 'internal_ops_queue'
    }
    return channel_map.get(action, 'unknown_channel')

In [10]:
def apply_agent(row):
    action, explanation = recommend_action(
        row['probability'],
        row['failure_reason_decoded'],
        row['customer_segment_decoded'],
        row['retry_count']
    )
    channel = map_action_to_channel(action)
    return pd.Series([action, explanation, channel])

X_test[['recommended_action', 'explanation', 'escalation_channel']] = X_test.apply(apply_agent, axis=1)

In [11]:
import datetime

X_test['decision_timestamp'] = datetime.datetime.now().isoformat()
X_test['audit_id'] = X_test.index.astype(str) + "_" + X_test['decision_timestamp'].str.replace(':', '-')

audit_trail = X_test[[
    'audit_id', 'decision_timestamp', 'probability', 'failure_reason_decoded',
    'customer_segment_decoded', 'retry_count', 'recommended_action',
    'escalation_channel', 'explanation'
]]

audit_trail.to_csv('../Data/audit_trail.csv', index=False)
print(f"Audit trail saved with {len(audit_trail)} logged decisions")

Audit trail saved with 1409 logged decisions


In [12]:
print(X_test.shape)
print('probability' in X_test.columns)

(1409, 25)
True


In [13]:
def recommend_action(probability, failure_reason, customer_segment, retry_count, max_retries=3):
    
    # STOPPING RULE — check this FIRST, overrides everything else below
    if retry_count >= max_retries:
        action = 'escalate_to_manual_review'
        explanation = f"Retry limit reached ({retry_count}/{max_retries} attempts). Stopping automated recovery and escalating to manual review to avoid excessive retry attempts."
        return action, explanation
    
    if failure_reason == 'card_expired':
        action = 'switch_payment_method'
        explanation = f"Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead."
    
    elif failure_reason == 'insufficient_funds' and probability >= 0.4:
        action = 'delayed_retry'
        explanation = f"Failure reason is insufficient_funds with a reasonable recovery probability ({probability:.2f}). Waiting allows time for the customer's balance to potentially refill before retrying."
    
    elif probability >= 0.7:
        action = 'immediate_retry'
        explanation = f"High recovery probability ({probability:.2f}) with no time-dependent failure reason. Retrying immediately is likely to succeed."
    
    elif probability < 0.5 and customer_segment == 'high_value_repeat':
        action = 'send_incentive'
        explanation = f"Recovery probability is moderate-to-low ({probability:.2f}), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment."
    
    else:
        action = 'delayed_retry'
        explanation = f"No strong signal for immediate action (probability={probability:.2f}). Defaulting to a low-cost delayed retry rather than an expensive incentive."
    
    return action, explanation

In [16]:
X_test[['recommended_action', 'explanation', 'escalation_channel']] = X_test.apply(apply_agent, axis=1)

In [18]:
pd.set_option('display.max_colwidth', None)
X_test[['probability', 'failure_reason_decoded', 'customer_segment_decoded', 'retry_count', 'recommended_action', 'explanation']].sample(5)

,probability,failure_reason_decoded,customer_segment_decoded,retry_count,recommended_action,explanation
620,0.673341,bank_decline,high_value_repeat,0,delayed_retry,No strong signal for immediate action (probability=0.67). Defaulting to a low-cost delayed retry rather than an expensive incentive.
36,0.666807,insufficient_funds,high_value_repeat,0,delayed_retry,Failure reason is insufficient_funds with a reasonable recovery probability (0.67). Waiting allows time for the customer's balance to potentially refill before retrying.
515,0.500538,insufficient_funds,occasional,1,delayed_retry,Failure reason is insufficient_funds with a reasonable recovery probability (0.50). Waiting allows time for the customer's balance to potentially refill before retrying.
971,0.617247,insufficient_funds,high_value_repeat,0,delayed_retry,Failure reason is insufficient_funds with a reasonable recovery probability (0.62). Waiting allows time for the customer's balance to potentially refill before retrying.
1100,0.581042,bank_decline,occasional,0,delayed_retry,No strong signal for immediate action (probability=0.58). Defaulting to a low-cost delayed retry rather than an expensive incentive.


In [19]:
X_test.to_csv('../Data/test_with_recommendations.csv', index=False)
print("Saved successfully")

Saved successfully


In [20]:
print('failure_reason_decoded' in X_test.columns)
print('customer_segment_decoded' in X_test.columns)
print('actual_recovered' in X_test.columns)

True
True
False


In [21]:
X_test['actual_recovered'] = y_test.values

In [22]:
print('actual_recovered' in X_test.columns)
print(X_test.shape)

True
(1409, 26)


In [23]:
print(X_test.columns.tolist())

['transaction_amount', 'customer_tenure_months', 'retry_count', 'gateway_response_time_ms', 'is_high_value_transaction', 'is_repeat_failure', 'failure_reason_bank_decline', 'failure_reason_card_expired', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'payment_method_Bank transfer (automatic)', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check', 'customer_segment_high_value_repeat', 'customer_segment_new_customer', 'customer_segment_occasional', 'customer_segment_decoded', 'failure_reason_decoded', 'probability', 'recommended_action', 'explanation', 'escalation_channel', 'decision_timestamp', 'audit_id', 'actual_recovered']


In [24]:
total_failed_revenue = X_test['transaction_amount'].sum()
print(f"Total revenue at stake: ₹{total_failed_revenue:,.2f}")

Total revenue at stake: ₹1,088,056.80


In [25]:
baseline_recovered = X_test[X_test['actual_recovered'] == 1]['transaction_amount'].sum()
print(f"Baseline (natural) recovery: ₹{baseline_recovered:,.2f}")
print(f"Baseline recovery rate: {baseline_recovered / total_failed_revenue:.1%}")

Baseline (natural) recovery: ₹550,431.00
Baseline recovery rate: 50.6%


In [26]:
# Transactions where the system recommends active intervention AND it didn't already recover naturally
recoverable_candidates = X_test[
    (X_test['actual_recovered'] == 0) &
    (X_test['recommended_action'].isin(['immediate_retry', 'delayed_retry', 'send_incentive']))
    # escalate_to_manual_review and switch_payment_method excluded — not direct "recovery" actions
]
potential_additional_revenue = recoverable_candidates['transaction_amount'].sum()
print(f"Additional revenue pool (system-flagged, not naturally recovered): ₹{potential_additional_revenue:,.2f}")

capture_rate = 0.40
additional_recovered = potential_additional_revenue * capture_rate
print(f"Estimated additional recovery at {capture_rate:.0%} capture rate: ₹{additional_recovered:,.2f}")

Additional revenue pool (system-flagged, not naturally recovered): ₹278,170.20
Estimated additional recovery at 40% capture rate: ₹111,268.08


In [27]:
total_recovered_with_system = baseline_recovered + additional_recovered
improvement_amount = additional_recovered
improvement_percentage = (additional_recovered / baseline_recovered) * 100

print(f"Baseline recovery (no system): ₹{baseline_recovered:,.2f}")
print(f"Total recovery WITH system: ₹{total_recovered_with_system:,.2f}")
print(f"Additional revenue recovered: ₹{improvement_amount:,.2f}")
print(f"Improvement over baseline: {improvement_percentage:.1f}%")

Baseline recovery (no system): ₹550,431.00
Total recovery WITH system: ₹661,699.08
Additional revenue recovered: ₹111,268.08
Improvement over baseline: 20.2%


In [28]:
card_expired_cases = X_test[X_test['failure_reason_decoded'] == 'card_expired']
wasted_retry_avoided = card_expired_cases['transaction_amount'].sum()
print(f"Transaction value where blind retries would be wasted: ₹{wasted_retry_avoided:,.2f}")
print(f"Number of such cases: {len(card_expired_cases)}")

Transaction value where blind retries would be wasted: ₹221,327.40
Number of such cases: 278


In [29]:
X_test.to_csv('../Data/test_with_full_analysis.csv', index=False)
print("Saved")

Saved


In [30]:
# Confirm stopping rule actually triggers for high retry counts
print(X_test[X_test['retry_count'] >= 3]['recommended_action'].value_counts())
# Should show 'escalate_to_manual_review' for all of these

# Confirm escalation channels are populated correctly
print(X_test['escalation_channel'].value_counts())

# Peek at the audit trail
print(audit_trail.head())

recommended_action
escalate_to_manual_review    150
Name: count, dtype: int64
escalation_channel
payment_gateway_scheduled_retry    795
sms_prompt_update_method           256
payment_gateway_auto_retry         188
internal_ops_queue                 150
sms_and_email_offer                 20
Name: count, dtype: int64
                       audit_id          decision_timestamp  probability  \
0  0_2026-08-31T12-11-54.525639  2026-08-31T12:11:54.525639     0.498491   
1  1_2026-08-31T12-11-54.525639  2026-08-31T12:11:54.525639     0.665660   
2  2_2026-08-31T12-11-54.525639  2026-08-31T12:11:54.525639     0.127500   
3  3_2026-08-31T12-11-54.525639  2026-08-31T12:11:54.525639     0.617793   
4  4_2026-08-31T12-11-54.525639  2026-08-31T12:11:54.525639     0.122896   

  failure_reason_decoded customer_segment_decoded  retry_count  \
0           bank_decline             new_customer            1   
1     insufficient_funds        high_value_repeat            0   
2           card_expired   

In [32]:
print(best_lr_model.feature_names_in_.tolist())

['transaction_amount', 'customer_tenure_months', 'retry_count', 'gateway_response_time_ms', 'is_high_value_transaction', 'is_repeat_failure', 'failure_reason_bank_decline', 'failure_reason_card_expired', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'payment_method_Bank transfer (automatic)', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check', 'customer_segment_high_value_repeat', 'customer_segment_new_customer', 'customer_segment_occasional']


In [33]:
total_failed_revenue = X_test['transaction_amount'].sum()
baseline_recovered = X_test[X_test['actual_recovered'] == 1]['transaction_amount'].sum()
...
total_recovered_with_system = baseline_recovered + additional_recovered
improvement_percentage = (additional_recovered / baseline_recovered) * 100
...
card_expired_cases = X_test[X_test['failure_reason_decoded'] == 'card_expired']
wasted_retry_avoided = card_expired_cases['transaction_amount'].sum()

In [34]:
import json

impact_summary = {
    "total_failed_revenue": round(float(total_failed_revenue), 2),
    "baseline_recovered": round(float(baseline_recovered), 2),
    "baseline_recovery_rate": round(float(baseline_recovered / total_failed_revenue), 4),
    "potential_additional_revenue": round(float(potential_additional_revenue), 2),
    "additional_recovered": round(float(additional_recovered), 2),
    "total_recovered_with_system": round(float(total_recovered_with_system), 2),
    "improvement_percentage": round(float(improvement_percentage), 2),
    "wasted_retry_avoided": round(float(wasted_retry_avoided), 2),
    "card_expired_cases_count": int(len(card_expired_cases))
}

with open('../Data/impact_summary.json', 'w') as f:
    json.dump(impact_summary, f, indent=2)

print("Impact summary saved")
print(impact_summary)

Impact summary saved
{'total_failed_revenue': 1088056.8, 'baseline_recovered': 550431.0, 'baseline_recovery_rate': 0.5059, 'potential_additional_revenue': 278170.2, 'additional_recovered': 111268.08, 'total_recovered_with_system': 661699.08, 'improvement_percentage': 20.21, 'wasted_retry_avoided': 221327.4, 'card_expired_cases_count': 278}
